# [1교시]

### 프롬프트 템플릿 Agent tool rag에서 표준
- role: LLM이 어떤 역할을 할 지 정함
- instruction: 답변 규칙과 형식을 정리
- context: 데이터베이스 검색 결과처럼 답변에 참고 할 실제 정보
- query:실제 질문

In [ ]:
# [코드 각주] 사용자 질문을 라우터, 도구 호출, LLM 최종 답변으로 연결하는 Agent 기본 흐름을 메모한 셀입니다.
# 사용자 질문
# 적절한 tool 호출 -> 라우터 ( Fast API )
# 수집한 정보를 -> LLM 전달
# 최종답변은 LLM

In [ ]:
# [코드 각주] OpenAI API 키를 환경 변수에서 불러오고, JSON 응답을 생성하는 OpenAILLM 래퍼 클래스를 정의합니다.
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
from dotenv import load_dotenv  # .env 파일의 API 키를 읽기 위해 dotenv를 사용합니다.
from openai import OpenAI  # OpenAI Chat Completions API를 사용하기 위한 클래스입니다.
load_dotenv(override=True)  # .env 값을 현재 실행 환경에 반영합니다.

api_key = os.getenv('OPENAI_API_KEY')  # 환경 변수에서 OpenAI API 키를 가져옵니다.
if not api_key:  # API 키가 없을 때 잘못된 실행을 막는 방어 코드입니다.
    raise EnvironmentError('openai api key .....')

class OpenAILLM:  # LLM 호출 코드를 재사용하기 쉽게 클래스로 묶습니다.
    def __init__(self,model:str = 'gpt-5.4-nano'):  # 모델명과 클라이언트 객체를 초기화합니다.
        self.client = OpenAI(api_key=api_key)  # API 요청을 보낼 클라이언트 객체를 저장합니다.
        self.model = model  # 사용할 모델명을 인스턴스 변수로 저장합니다.
    def generate(self, prompt:str) -> str:  # 프롬프트를 입력받아 LLM 응답 텍스트를 반환합니다.
        response = self.client.chat.completions.create(  # Chat Completions 방식으로 모델에 메시지를 전달합니다.
            model = self.model,
            messages =[  # system/user 역할 메시지를 모델에 전달하는 부분입니다.
                {"role":"system", "content":"You are an ecomerce recommendation assistant, Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,  # 생성 결과의 무작위성을 조절합니다. 0에 가까울수록 안정적입니다.
            response_format={'type':'json_object'}  # 모델 응답을 JSON 객체 형식으로 제한합니다.
        )
        return response.choices[0].message.content  # 모델이 생성한 최종 텍스트만 꺼내 반환합니다.
llm = OpenAILLM()
llm.model  

In [ ]:
# [코드 각주] 추천 답변에 사용할 실제 컨텍스트 역할의 상품 목록과 context 문자열을 구성합니다.
# 데이터
do_search_results = [  # LLM이 참고할 컨텍스트 데이터 목록입니다.
    {"id":"p1", "name":"고성능 노트북","category":"가전"},
    {"id":"p2", "name":"사무용 랩탑","category":"가전"},
    {"id":"p3", "name":"미러리스 카메라","category":"가전"},
]
context_string = ""  # 검색 결과를 프롬프트에 넣기 위한 문자열로 누적합니다.
for item in do_search_results:  # 상품 목록을 하나씩 돌면서 프롬프트용 context를 만듭니다.
    context_string += f"- 상품명:{item['name']} | 카테고리:{item['category']}"

In [ ]:
# [코드 각주] role, instruction, context, query 구조를 반영한 추천 프롬프트 템플릿을 생성합니다.
# 프롬프트 템플릿 - 고정된 구조
from  textwrap import dedent  #  들여쓰기 indent를 제거
user_query = '코딩할 때 쓸만한 노트북 하나 추천해줘, 믿을만한 제품으로'  # 사용자가 실제로 입력한 질문 예시입니다.
raw_prompt = dedent(f'''
                    당신은 이커머스 전문 AI 추천 어시스턴트입니다.
                    아래 컨텍스트를 참고하여 사용자의 질문에 답하세요
                    반드시 json 형식으로만 답하세요

                    [컨텍스트]
                    {context_string}

                    [질문]
                    {user_query}

                    답변은 반드시 아래 형식의 json으로만 출력하세요
                    {
                        {
                            "recommended_product" : "상품명",
                            "reason":"추천사유"                            
                        }
                    }
                    ''')
print(raw_prompt)

## LLM 응답

In [ ]:
# [코드 각주] LLM 응답을 JSON으로 파싱하고 사람이 읽기 좋게 출력합니다.
import json
response = llm.generate(raw_prompt)  # 생성한 프롬프트를 LLM에 전달합니다.
data = json.loads(response)  # 문자열 JSON 응답을 파이썬 딕셔너리로 바꿉니다.
print(json.dumps(data, indent=2, ensure_ascii=False))

# [2교시]

## 라우팅
- 들어온 질문을 보고 어느 경로로 보낼지 결정하는 단계

In [ ]:
# [코드 각주] 질문의 키워드를 기준으로 뉴스, 계산, 메모리, 추천, 일반 LLM 경로를 분류합니다.
def route_question(question:str)->str:  # 사용자 질문의 의도에 맞는 처리 경로를 결정합니다.
    lower_question = question.lower()  # 키워드 비교를 쉽게 하려고 질문을 소문자로 변환합니다.
    if any(keyword in lower_question for keyword in ['뉴스','기사','검색','찾아줘','최신','오늘']):  # 질문에 특정 키워드가 포함됐는지 검사합니다.
        return "news_tool"
    elif any(keyword in lower_question for keyword in ['계산','더하기','곱하기','합계','몇','얼마']):  # 앞 조건이 아니면 다음 도구 후보를 검사합니다.
        return "calculator_tool"
    elif any(keyword in lower_question for keyword in ['기억','기록','선호','메모','이전']):  # 앞 조건이 아니면 다음 도구 후보를 검사합니다.
        return "memory_tool"
    elif any(keyword in lower_question for keyword in ['추천','골라','비교']):  # 앞 조건이 아니면 다음 도구 후보를 검사합니다.
        return "llm_recomendation"
    else:
        return "general_llm"
sample_questions = [  # 라우터 테스트에 사용할 질문 목록입니다.
    "3개 상품을 2개씩 주문하면 총 몇 개인가?"
    "나는 코딩용 노트북을 좋아한다는 점을 기억해줘."
    "AI 에이전트 뉴스 최신 기사 3개 찾아줘"
    "코딩할 때 쓸만한 노트북 추천해줘."
]
for question in sample_questions:
    print(f'질문 : {question} | 라우트 : {route_question(question)}')

## tool 활용

In [ ]:
# [코드 각주] 계산 도구와 네이버 뉴스 도구의 뼈대를 작성합니다.
import json
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
import re
from urllib.parse import quote
from urllib.request import Request,urlopen


def calculator_tool(text:str)->float:  # 문자열 계산식을 받아 계산 결과를 반환하는 도구입니다.
    allowed_chars = set("0123456789+-*/().")  # eval 실행 전 허용할 문자만 제한해 위험한 입력을 줄입니다.
    if not set(text) <= allowed_chars: # text의 문자 중에 허용되지 않은 문자가 있다면
        raise valueError('허용되지 않은 문자가 포함되어 있습니다.')
    return eval(text)  # 검증된 계산식을 파이썬 표현식으로 계산합니다.

def search_naver_news(query:str, display:int=3) -> list[dict]:  # 네이버 뉴스 검색 API를 호출하는 도구 함수입니다.
    pass

In [ ]:
# [코드 각주] 네이버 뉴스 검색 API를 호출하고 HTML 태그 제거, 날짜 정리, 링크 추출을 수행합니다.
# 네이버 검색 API 예제 - 블로그 검색
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv  # .env 파일의 API 키를 읽기 위해 dotenv를 사용합니다.
load_dotenv(override=True)  # .env 값을 현재 실행 환경에 반영합니다.

def _format_date(pubdate):  # 네이버 pubDate 문자열을 YYYY-MM-DD 형태로 정리합니다.
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):  # 뉴스 제목/본문에서 HTML 태그와 엔티티를 제거합니다.
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')  # 네이버 API Client ID를 환경 변수에서 읽습니다.
client_secret = os.getenv('NAVER_CLIENT_SECRET')  # 네이버 API Client Secret을 환경 변수에서 읽습니다.

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:  # 네이버 뉴스 검색 API를 호출하는 도구 함수입니다.
    encText = urllib.parse.quote(query)  # 검색어를 URL에 넣을 수 있도록 인코딩합니다.
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)  # URL과 헤더를 포함한 HTTP 요청 객체를 만듭니다.
    request.add_header("X-Naver-Client-Id",client_id)  # 네이버 API 인증용 Client ID/Secret을 헤더에 넣습니다.
    request.add_header("X-Naver-Client-Secret",client_secret)  # 네이버 API 인증용 Client ID/Secret을 헤더에 넣습니다.


    response = urllib.request.urlopen(request)  # API 서버에 요청을 보내고 응답을 받습니다.
    rescode = response.getcode()
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)  # API 응답 문자열을 JSON 객체로 변환합니다.

        for row in result.get('items'):
            items.append({  # 뉴스 제목, 요약, 날짜, 링크를 결과 리스트에 추가합니다.
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [ ]:
# [코드 각주] 네이버 뉴스 도구가 특정 키워드로 동작하는지 확인합니다.
search_naver_news('AI 에이전트')

## 메모리 활용하기
- 이전대화나 사용자의 선호를 저장해서 다음 응답에 반영하는 기능

In [ ]:
# [코드 각주] 사용자 선호나 이전 질문을 세션 메모리에 저장하고 조회하는 구조를 만듭니다.
session_memory={}  # 간단한 세션 메모리를 딕셔너리로 만듭니다.
def remember_preference(user_id:str, key:str, value:str)->None:  # 사용자별 선호 정보를 메모리에 저장합니다.
    if user_id not in session_memory:
        session_memory[user_id] = {}
    session_memory[user_id][key] = value

def get_preference(user_id:str, key:str, default:str | None = None) -> str | None:  # 저장된 사용자 선호 정보를 조회합니다.
    return session_memory.get(user_id,{}).get(key,default)

user_id = 'student-001'  # 메모리 저장 대상 사용자 ID 예시입니다.
remember_preference(user_id, 'category', '노트북')  # 사용자의 선호 정보를 key-value 형태로 저장합니다.
remember_preference(user_id, 'usage', '코딩')  # 사용자의 선호 정보를 key-value 형태로 저장합니다.

print(session_memory)

# [3교시]

### 프롬프트 템플릿  Agent tool rag에서 표준
- role:LLM이 어떤 역활을 할지 정함
- instruction:답변 규칙과 형식을 정리
- context:데이터베이스 검색결과처럼 답변에 참고할 실제 정보
- query:실제 질문


In [ ]:
# [코드 각주] 3교시 Agent 구성 흐름을 다시 정리한 메모성 코드 셀입니다.
# 사용자 질문 -> LLM개입해서 사용자 의도를 파악해서
# 적절한 tool 호출 -> 라우터( Fast API )
# 수집한 정보를 -> LLM 전달
# 최종답변은 LLM

In [ ]:
# [코드 각주] OpenAI 기반 LLM 호출 클래스를 다시 정의합니다.
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
from dotenv import load_dotenv  # .env 파일의 API 키를 읽기 위해 dotenv를 사용합니다.
from openai import OpenAI  # OpenAI Chat Completions API를 사용하기 위한 클래스입니다.
load_dotenv(override=True)  # .env 값을 현재 실행 환경에 반영합니다.

api_key = os.getenv('OPENAI_API_KEY')  # 환경 변수에서 OpenAI API 키를 가져옵니다.
if not api_key:  # API 키가 없을 때 잘못된 실행을 막는 방어 코드입니다.
    raise EnvironmentError('openai api key .....')

class OpenAILLM:  # LLM 호출 코드를 재사용하기 쉽게 클래스로 묶습니다.
    def __init__(self,model:str = 'gpt-5.4-nano'):  # 모델명과 클라이언트 객체를 초기화합니다.
        self.client = OpenAI(api_key=api_key)  # API 요청을 보낼 클라이언트 객체를 저장합니다.
        self.model = model  # 사용할 모델명을 인스턴스 변수로 저장합니다.
    def generate(self, prompt:str) -> str:  # 프롬프트를 입력받아 LLM 응답 텍스트를 반환합니다.
        response = self.client.chat.completions.create(  # Chat Completions 방식으로 모델에 메시지를 전달합니다.
            model = self.model,
            messages =[  # system/user 역할 메시지를 모델에 전달하는 부분입니다.
                {"role":"system", "content":"You are an ecomerce recommendation assistant, Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,  # 생성 결과의 무작위성을 조절합니다. 0에 가까울수록 안정적입니다.
            response_format={'type':'json_object'}  # 모델 응답을 JSON 객체 형식으로 제한합니다.
        )
        return response.choices[0].message.content  # 모델이 생성한 최종 텍스트만 꺼내 반환합니다.
llm = OpenAILLM()
llm.model  

In [ ]:
# [코드 각주] 추천/응답 생성을 위한 컨텍스트 데이터를 다시 구성합니다.
# 데이터
do_search_results = [  # LLM이 참고할 컨텍스트 데이터 목록입니다.
    {"id":"p1", "name":"고성능 노트북","category":"가전"},
    {"id":"p2", "name":"사무용 랩탑","category":"가전"},
    {"id":"p3", "name":"미러리스 카메라","category":"가전"},
]
context_string = ""  # 검색 결과를 프롬프트에 넣기 위한 문자열로 누적합니다.
for item in do_search_results:  # 상품 목록을 하나씩 돌면서 프롬프트용 context를 만듭니다.
    context_string += f"- 상품명:{item['name']} | 카테고리:{item['category']}"

In [ ]:
# [코드 각주] 사용자 질문과 context를 넣어 JSON 답변을 반환하는 recommend_product 함수를 정의합니다.
from  textwrap import dedent  #  들여쓰기 indent를 제거
import json
# 프롬프트 템플릿 - 고정된 구조
# 프롬프트생성 + llm호출 + 파싱
def recommend_product(user_question:str, context:str) ->dict:    
    prompt = dedent(f'''
                        당신은  사용자의 질문에 정확히 응답하는 ai 시스템 입니다.
                        사용자의 질문과 context를 보고 질문의 의도에 맞게 출력하세요

                        [컨텍스트]
                        {context}

                        [질문]
                        {user_question}                        
                        
                        [출력]
                        답변은 반드시 아래와 같은 json형태로 
                        {
                            {
                                "assistant" : "출력내용",
                                "reason":"사유"                            
                            }
                        }

                        ''')
    response = llm.generate(prompt)  # 생성한 프롬프트를 LLM에 전달합니다.
    data = json.loads(response)  # 문자열 JSON 응답을 파이썬 딕셔너리로 바꿉니다.
    return json.dumps(data, indent=2, ensure_ascii=False)
    
    

### 라우팅
- 들어온 질문을 보고 어느 경로로 보낼지 결정하는 단계

In [ ]:
# [코드 각주] 라우팅 함수를 개선하여 각 질문을 tool 사용 경로로 분류합니다.
def route_qeustion(question:str)->str:  # 사용자 질문의 의도에 맞는 처리 경로를 결정합니다.
    lower_question = question.lower()  # 키워드 비교를 쉽게 하려고 질문을 소문자로 변환합니다.
    if any(keyword in lower_question for keyword in ['뉴스','기사','검색','찾아줘','최신','오늘']):  # 질문에 특정 키워드가 포함됐는지 검사합니다.
        return "news_tool"
    elif any(keyword in lower_question for keyword in ['계산','더하기','곱하기','합계','몇','얼마']):  # 앞 조건이 아니면 다음 도구 후보를 검사합니다.
        return "calculator_tool"
    elif any(keyword in lower_question for keyword in ['기억','기록','선호','메모','이전']):  # 앞 조건이 아니면 다음 도구 후보를 검사합니다.
        return "memory_tool"
    elif any(keyword in lower_question for keyword in ['추천','골라','비교']):  # 앞 조건이 아니면 다음 도구 후보를 검사합니다.
        return "llm_recommendation"
    else:
        return "general_llm"
sample_questions = [  # 라우터 테스트에 사용할 질문 목록입니다.
    "3개 상품을 2개씩 주문하면 총 몇 개인가?",
    "나는 코딩용 노트북을 좋아한다는 점을 기억해줘.",
    "AI 에이전트 뉴스 최신 기사 3개 찾아줘.",
    "코딩할 때 쓸만한 노트북 추천해줘.",
]
for question in sample_questions:
    print(f'질문 : {question} | 라우트 : {route_qeustion(question)}')    

## tool 활용

In [ ]:
# [코드 각주] 계산식을 안전하게 검사한 뒤 계산하는 calculator tool을 정의합니다.
import json
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
import re
from urllib.parse import quote
from urllib.request import Request,urlopen


def calcualtor_tool(text:str)->float:  # 문자열 계산식을 받아 계산 결과를 반환하는 도구입니다.
    allowed_chars = set("0123456789+-*/(). ")  # eval 실행 전 허용할 문자만 제한해 위험한 입력을 줄입니다.
    if not set(text) <= allowed_chars: # text의 문자중에 허용되지 않은 문자가 있다면
        raise ValueError('허용되지 않은 문자가 포함되어 있습니다.')
    return eval(text)  # 검증된 계산식을 파이썬 표현식으로 계산합니다.

In [ ]:
# [코드 각주] 네이버 뉴스 검색 API를 실제 도구 함수로 구현합니다.
# 네이버 검색 API 예제 - 블로그 검색
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv  # .env 파일의 API 키를 읽기 위해 dotenv를 사용합니다.
load_dotenv(override=True)  # .env 값을 현재 실행 환경에 반영합니다.

def _format_date(pubdate):  # 네이버 pubDate 문자열을 YYYY-MM-DD 형태로 정리합니다.
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):  # 뉴스 제목/본문에서 HTML 태그와 엔티티를 제거합니다.
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')  # 네이버 API Client ID를 환경 변수에서 읽습니다.
client_secret = os.getenv('NAVER_CLIENT_SECRET')  # 네이버 API Client Secret을 환경 변수에서 읽습니다.

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:  # 네이버 뉴스 검색 API를 호출하는 도구 함수입니다.
    encText = urllib.parse.quote(query)  # 검색어를 URL에 넣을 수 있도록 인코딩합니다.
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)  # URL과 헤더를 포함한 HTTP 요청 객체를 만듭니다.
    request.add_header("X-Naver-Client-Id",client_id)  # 네이버 API 인증용 Client ID/Secret을 헤더에 넣습니다.
    request.add_header("X-Naver-Client-Secret",client_secret)  # 네이버 API 인증용 Client ID/Secret을 헤더에 넣습니다.


    response = urllib.request.urlopen(request)  # API 서버에 요청을 보내고 응답을 받습니다.
    rescode = response.getcode()
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)  # API 응답 문자열을 JSON 객체로 변환합니다.

        for row in result.get('items'):
            items.append({  # 뉴스 제목, 요약, 날짜, 링크를 결과 리스트에 추가합니다.
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [ ]:
# [코드 각주] 계산 도구와 뉴스 검색 도구를 직접 호출해 결과를 확인합니다.
print(calcualtor_tool('30*50'))
search_naver_news('AI 에이전트')

### 메모리 활용하기
- 이전대화나 사용자의 선호를 저장해서 다음 응답에 반영하는 기능

In [ ]:
# [코드 각주] 메모리 저장/조회 기능을 다시 구성하고 student-001의 선호를 저장합니다.
session_memory={}  # 간단한 세션 메모리를 딕셔너리로 만듭니다.
def remember_preference(user_id:str, key:str, value:str)->None:  # 사용자별 선호 정보를 메모리에 저장합니다.
    if user_id not in session_memory:
        session_memory[user_id] = {}
    session_memory[user_id][key] = value

def get_preference(user_id:str, key:str, default:str | None = None) -> str | None:  # 저장된 사용자 선호 정보를 조회합니다.
    return session_memory.get(user_id,{}).get(key,default)

user_id = 'student-001'  # 메모리 저장 대상 사용자 ID 예시입니다.
remember_preference(user_id, 'category','노트북')  # 사용자의 선호 정보를 key-value 형태로 저장합니다.
remember_preference(user_id, 'usage','코딩')  # 사용자의 선호 정보를 key-value 형태로 저장합니다.

print(session_memory)

### 라우터 + 도구 + 메모리 통합
- 라우팅 규칙, 계산도구, 네이버뉴스도구, 메모리 저장을 하나의 흐름으로 연결--> Agent의 기본 형태
- 먼저 질문을 분류하고 그 분류 결과에 맞는 도구를 호출한뒤 필요하면 메모리까지 갱신
- 최종 결과를 llm에 전달해서 답변을 생성

In [ ]:
# [코드 각주] 질문에서 계산식과 뉴스 검색어를 추출하고, 라우팅 결과에 따라 도구를 실행하는 통합 함수를 만듭니다.
def extract_math_expression(question: str) -> str:  # 질문에서 계산식 부분만 뽑아냅니다.
    match = re.search(r"[0-9\s\+\-\*\/\(\)\.]+", question)  # 정규표현식으로 계산식 후보를 찾습니다.
    if not match:
        raise ValueError("계산식을 찾을 수 없습니다.")
    expression = match.group(0).strip()
    if not expression:
        raise ValueError("계산식이 비어 있습니다.")
    return expression

def extract_news_query(question: str) -> str:  # 질문에서 뉴스 검색에 필요한 키워드만 정리합니다.
    cleaned = re.sub(r"뉴스|기사|검색|알려줘|찾아줘|추천해줘|좀|최근|최신|오늘", " ", question)  # 불필요한 단어와 특수문자를 제거합니다.
    cleaned = re.sub(r"\d+\s*개?", " ", cleaned)  # 불필요한 단어와 특수문자를 제거합니다.
    cleaned = re.sub(r"[^0-9A-Za-z가-힣\s]", " ", cleaned)  # 불필요한 단어와 특수문자를 제거합니다.
    cleaned = re.sub(r"\s+", " ", cleaned).strip()  # 불필요한 단어와 특수문자를 제거합니다.
    return cleaned or question

def route_and_execute(question: str) -> dict:  # 사용자 질문의 의도에 맞는 처리 경로를 결정합니다.
    route = route_qeustion(question)  # 질문이 어떤 도구로 가야 하는지 먼저 분류합니다.

    if route == "news_tool":
        news_query = extract_news_query(question)  # 뉴스 API에 넣을 검색어를 추출합니다.
        news_result = search_naver_news(news_query, display=3)  # 뉴스 도구 실행 결과를 저장합니다.
        news_result = recommend_product('뉴스 요약해줘',news_result)  # 뉴스 도구 실행 결과를 저장합니다.
        return {
            "route": route,
            "tool": "search_naver_news",
            "input": news_query,
            "result": news_result,
        }

    if route == "calculator_tool":
        expression = extract_math_expression(question)
        result = calcualtor_tool(expression)
        return {
            "route": route,
            "tool": "calculator_tool",
            "input": expression,
            "result": result,
        }

    if route == "memory_tool":
        remember_preference("student-001", "last_question", question)  # 사용자의 선호 정보를 key-value 형태로 저장합니다.
        return {
            "route": route,
            "tool": "memory_tool",
            "input": question,
            "result": get_preference("student-001", "last_question"),
        }

    if route == "llm_recommendation":
        recommendation = recommend_product(question, context_string)  # LLM 추천 결과를 저장합니다.
        return {
            "route": route,
            "tool": "llm_recommendation",
            "input": question,
            "result": recommendation,
        }

    return {
        "route": route,
        "tool": "general_llm",
        "input": question,
        "result": question,
    }

In [ ]:
# [코드 각주] route_and_execute가 뉴스 질문을 어떻게 처리하는지 확인합니다.
demo_questions = [
    # '3 + 2 * 4는 얼마야',
    '어제 오늘 사고뉴스 5개 찾아줘',
    '점심메뉴 추천해줘',
    '연구용 노트북 추천해줘',
]
print(json.loads(route_and_execute('어제 오늘 사고뉴스 5개 찾아줘')['result'])['assistant'])
print(json.loads(route_and_execute('어제 오늘 사고뉴스 5개 찾아줘')['result'])['reason'])
# for question in demo_questions:
#     result = route_and_execute(question)        
#     print(f'\n질문:{question}')
#     print(f'라우트:{result["route"]}')
#     print(f'도구:{result["tool"]}')
#     print(f'결과:{result["result"]}')
#     print(f'결과:{result["result"]["reason"]}')

In [ ]:
# [코드 각주] 뉴스 출력과 추천 프롬프트 개선 방향을 메모합니다.
# 1. 뉴스검색의 경우... 출력을 조정
# 2. 추천의 경우, 프롬프트가 현재 이커머스로 되어있는데 -> 일반적인 프롬프트로 변경

# [4교시]

In [ ]:
# [코드 각주] 복합 질문 처리를 위해 OpenAI 기반 LLM 호출 객체를 다시 준비합니다.
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
from dotenv import load_dotenv  # .env 파일의 API 키를 읽기 위해 dotenv를 사용합니다.
from openai import OpenAI  # OpenAI Chat Completions API를 사용하기 위한 클래스입니다.
load_dotenv(override=True)  # .env 값을 현재 실행 환경에 반영합니다.

api_key = os.getenv('OPENAI_API_KEY')  # 환경 변수에서 OpenAI API 키를 가져옵니다.
if not api_key:  # API 키가 없을 때 잘못된 실행을 막는 방어 코드입니다.
    raise EnvironmentError('openai api key .....')
class OpenAILLM:  # LLM 호출 코드를 재사용하기 쉽게 클래스로 묶습니다.
    def __init__(self,model:str = 'gpt-5.4-nano'):  # 모델명과 클라이언트 객체를 초기화합니다.
        self.client = OpenAI(api_key=api_key)  # API 요청을 보낼 클라이언트 객체를 저장합니다.
        self.model = model  # 사용할 모델명을 인스턴스 변수로 저장합니다.
    def generate(self, prompt:str) -> str:  # 프롬프트를 입력받아 LLM 응답 텍스트를 반환합니다.
        response = self.client.chat.completions.create(  # Chat Completions 방식으로 모델에 메시지를 전달합니다.
            model = self.model,
            messages =[  # system/user 역할 메시지를 모델에 전달하는 부분입니다.
                {"role":"system", "content":"너는 사용자의 질문에 친철하고 정확하게 답변하는 시스템 입니다., Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,              # 생성 결과의 무작위성을 조절합니다. 0에 가까울수록 안정적입니다.
        )
        return response.choices[0].message.content  # 모델이 생성한 최종 텍스트만 꺼내 반환합니다.
llm = OpenAILLM()    

In [ ]:
# [코드 각주] LLM이 외부 도구 없이 주식 관련 질문에 어떻게 답하는지 확인합니다.
llm.generate('어제 삼성전자 주식 종가 알려줘')

In [ ]:
# [코드 각주] 복합 질문을 여러 의도로 분리하는 routerLLM 함수를 작성합니다.
import json
from  textwrap import dedent  #  들여쓰기 indent를 제거
query = '어제 삼성전자 주식종가 를 조회하고 해당 종가와 비슷한종목을 뉴스에서 검색해서 요약하고 오늘날씨에 맞는 외출복을 추천하고 데이트 장소 추천해'
def routerLLM(query):  # 복합 질문을 여러 의도로 분리하는 LLM 라우터입니다.
    prompt = dedent(f"""
    사용자의 질문 의도를 분석하세요.

    질문에 포함된 의도가 여러 개이면
    반드시 모든 의도를 각각 분리하여 출력하세요.

    예를 들어:
    - 주식 + 뉴스 + 날씨
    - 뉴스 + 검색
    - 계산 + 주식

    처럼 복합 질문이면
    반드시 여러 개의 JSON 객체를 배열로 출력해야 합니다.

    question_type 별로 tool_query를 생성하세요.
    tool_query는 반드시 키워드중심으로 생성하세요 llm에 전달하는 문구가 아님을 명심해서 추론을하지말고 검색용 키워드로 매칭해주세요    
    뉴스는 api검색할수 있는 키워드중심으로,
    주식은 종목명만 추출하세요             
    날씨도 검색용 키워드로 추출하세요                               

    지원 가능한 question_type 예시:
    - 날씨
    - 뉴스
    - 주식
    - 검색
    - 계산
    - 지도

    [중요 규칙]
    - 질문에 포함된 모든 의도를 누락 없이 출력
    - 반드시 JSON 배열(Array) 형식으로 출력
    - 하나만 출력 금지
    - 설명문 금지
    - markdown 금지
    - ```json 금지

    [예시 입력]
    어제 삼성전자 종가 알려주고 관련 뉴스와 오늘 날씨도 알려줘

    [예시 출력]
    [
        {{
            "question_type": "주식",
            "tool_query": "삼성전자",
            "reason": "삼성전자 종가 요청"
        }},
        {{
            "question_type": "뉴스",
            "tool_query": "삼성전자 관련 최근 뉴스 검색",
            "reason": "관련 뉴스 요청"
        }},
        {{
            "question_type": "날씨",
            "tool_query": "오늘 날씨 조회",
            "reason": "날씨 요청"
        }}
    ]

    [질문]
    {query}

    [출력]
    반드시 유효한 JSON 배열만 출력하세요.
    """)

    result = llm.generate(prompt)
    json_result = json.loads(result)  # LLM 라우팅 결과를 JSON 배열로 파싱합니다.
    return json_result

In [ ]:
# [코드 각주] router 결과에서 나온 뉴스 검색어를 실제 네이버 뉴스 API에 넣기 위한 도구를 정의합니다.
# 네이버 검색 API 예제 - 블로그 검색
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv  # .env 파일의 API 키를 읽기 위해 dotenv를 사용합니다.
load_dotenv(override=True)  # .env 값을 현재 실행 환경에 반영합니다.

def _format_date(pubdate):  # 네이버 pubDate 문자열을 YYYY-MM-DD 형태로 정리합니다.
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):  # 뉴스 제목/본문에서 HTML 태그와 엔티티를 제거합니다.
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')  # 네이버 API Client ID를 환경 변수에서 읽습니다.
client_secret = os.getenv('NAVER_CLIENT_SECRET')  # 네이버 API Client Secret을 환경 변수에서 읽습니다.

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:  # 네이버 뉴스 검색 API를 호출하는 도구 함수입니다.
    encText = urllib.parse.quote(query)  # 검색어를 URL에 넣을 수 있도록 인코딩합니다.
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)  # URL과 헤더를 포함한 HTTP 요청 객체를 만듭니다.
    request.add_header("X-Naver-Client-Id",client_id)  # 네이버 API 인증용 Client ID/Secret을 헤더에 넣습니다.
    request.add_header("X-Naver-Client-Secret",client_secret)  # 네이버 API 인증용 Client ID/Secret을 헤더에 넣습니다.


    response = urllib.request.urlopen(request)  # API 서버에 요청을 보내고 응답을 받습니다.
    rescode = response.getcode()    
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)          # API 응답 문자열을 JSON 객체로 변환합니다.
        for row in result.get('items'):
            items.append({  # 뉴스 제목, 요약, 날짜, 링크를 결과 리스트에 추가합니다.
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [ ]:
# [코드 각주] 삼성전자 관련 뉴스 검색어로 뉴스 도구를 테스트합니다.
search_naver_news('삼성전자 관련 최근 뉴스 검색')

# [5교시]

In [ ]:
# [코드 각주] router 결과 중 뉴스 의도만 골라 도구를 실행하는 흐름을 확인합니다.
for row in json_result:
    if row.get('question_type') == '뉴스':
        print('검색어', row.get('tool_query'))
        print(search_naver_news(row.get('tool_query')))

In [ ]:
# [코드 각주] router_result를 순회하며 question_type에 맞는 도구를 실행하고 결과를 모읍니다.
# tool excution
tool_results = []  # 각 도구 실행 결과를 모으는 리스트입니다.
def excute_tools(router_result):  # 라우터 결과를 순회하며 필요한 도구를 실행합니다.
    for row in router_result:
        query_type = row.get('question_type')  # 현재 실행할 도구의 종류를 꺼냅니다.
        print(f'tool : {query_type}')
        if query_type == '뉴스':        
            query = row.get('tool_query')
            news_result = search_naver_news(row.get('tool_query'))  # 뉴스 도구 실행 결과를 저장합니다.
            tool_results.append({
                'question_type' : '뉴스',
                'query' : query,
                'result' : news_result
            })
        # other tools
    return tool_results            

In [ ]:
# [코드 각주] 여러 도구 결과를 최종 LLM 프롬프트로 합쳐 답변을 생성합니다.
def make_message(user_query:str, tool_results:list[dict]):  # 최종 LLM에 전달할 system/user 메시지를 구성합니다.
    prompt = f'''
    너는 다양한 도구의 결과를 종합해서 사용자 질문에 답변하는 ai assistant 입니다.

    [사용자질문]
    {user_query}

    [도구실행결과]
    {json.dumps(tool_results, ensure_ascii=False, indent=2)}

    [지침]
    - tool 결과를 기반으로 답변하세요
    - 필요한 경우 여러 tool 결과를 종합하세요
    - 지도 주식 날씨 검색 추천등 다양한 데이터가 포함될수 있습니다.
    - tool 결과내에 있는 내용에 우선순위를 높게해서 해당 데이터기반으로 답변하세요
    '''
    return [
                {"role":"system", "content":"너는 여러 tool결과를 조합해서 최종 답변을생성하는 ai agent 입니다."},
                {"role":"user", "content":prompt}
            ]
    
    
# final llm
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))  # 최종 답변 생성에 사용할 OpenAI 클라이언트입니다.
def final_llm_openai(user_query:str, tool_results:list[dict]):      # 도구 결과를 OpenAI 모델로 종합해 최종 답변을 만듭니다.
    result = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages =make_message(user_query,tool_results),  # system/user 역할 메시지를 모델에 전달하는 부분입니다.
            temperature=0.3              # 생성 결과의 무작위성을 조절합니다. 0에 가까울수록 안정적입니다.
        )
    return result.choices[0].message.content

### 파이프라인
- 사용자 질문
- 질문의 의도를 분석하는 llm 호출
- 해당 tools을 호출해서 결과 수집
- 최종 llm에게 전달
- 최종 답변 생성

In [ ]:
# [코드 각주] 사용자 질문 → router → tool execution → final LLM의 전체 파이프라인을 실행합니다.
query = '붕괴사고에 대해서'
router_result = routerLLM(query)  # 사용자 질문을 라우터에 넣어 의도를 분리합니다.
tool_results = excute_tools(router_result)  # 각 도구 실행 결과를 모으는 리스트입니다.
final_result = final_llm_openai(query,tool_results)  # 최종 답변 생성 결과를 저장합니다.
print(final_result)

# [6교시]

In [ ]:
# [코드 각주] Agent 구조를 planner/router, tool executor, memory/retrieval, LLM, final response로 정리합니다.
# 1. planner / router
# 2. tool excuter
# 3. memory / retrieval
# 4. LLM
# 5. final response

### 모델을 허깅페이스계열의 모델로 변경

In [ ]:
# [코드 각주] Hugging Face 계열 모델 사용을 위해 노트북 로그인을 수행합니다.
from huggingface_hub import notebook_login  # Hugging Face 로그인을 노트북에서 수행하기 위한 함수입니다.

os.environ.pop('HF_TOKEN', None)  # 기존 HF_TOKEN 환경 변수를 제거해 새 로그인 상태를 사용합니다.
notebook_login()  # Hugging Face 토큰 로그인을 실행합니다.

In [ ]:
# [코드 각주] Hugging Face 토큰을 확인하고 로그인 사용자를 출력합니다.
from huggingface_hub import get_token, whoami  # 저장된 Hugging Face 토큰과 사용자 정보를 확인합니다.

HF_TOKEN = get_token()  # 현재 세션에 저장된 Hugging Face 토큰을 가져옵니다.
if not HF_TOKEN:
    raise RuntimeError('Hugging Face 토큰이 저장되지 않았습니다. 로그인 셀을 다시 실행하세요.')

info = whoami(token=HF_TOKEN)  # 토큰으로 로그인 사용자를 확인합니다.
print('logged in user:', info.get('name'))
print('token prefix:', HF_TOKEN[:6] + '***')

### 모델 변경

In [ ]:
# [코드 각주] Qwen2.5-0.5B-Instruct 모델과 토크나이저를 AutoModel/AutoTokenizer로 불러옵니다.
from transformers import AutoModelForCausalLM, AutoTokenizer  # Causal LM 모델과 토크나이저를 자동으로 불러옵니다.
model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # 사용할 Hugging Face 모델 ID입니다.
model = AutoModelForCausalLM.from_pretrained(  # 사전학습 언어모델을 다운로드/로드합니다.
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)  # 모델에 맞는 토크나이저를 로드합니다.

# [7교시]

In [ ]:
# [코드 각주] tool 결과를 Qwen 채팅 템플릿에 맞게 넣고 최종 답변을 생성합니다.
def final_llm_qween(user_query:str, tool_results:list[dict]):    
    text = tokenizer.apply_chat_template(  # messages를 모델이 이해하는 채팅 템플릿 문자열로 바꿉니다.
        make_message(user_query,tool_results),
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)  # 텍스트를 토큰 ID 텐서로 변환하고 모델 장치로 보냅니다.
    generated_ids = model.generate(  # 모델이 이어서 생성할 토큰을 만듭니다.
        **model_inputs,
        max_new_tokens=512  # 새로 생성할 최대 토큰 수를 제한합니다.
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]  # 생성된 토큰 ID를 사람이 읽는 문자열로 복원합니다.
    return response

In [ ]:
# [코드 각주] 붕괴사고 질문을 Qwen 기반 final LLM 흐름으로 처리합니다.
query = '붕괴사고에 대해서'
router_result = routerLLM(query)  # 사용자 질문을 라우터에 넣어 의도를 분리합니다.
tool_results = excute_tools(router_result)    # 각 도구 실행 결과를 모으는 리스트입니다.
final_result = final_llm_qween(query,tool_results)  # 최종 답변 생성 결과를 저장합니다.
print(final_result)

In [ ]:
# [코드 각주] 최신 뉴스 요약 질문을 Qwen 기반 final LLM 흐름으로 처리합니다.
query = '오늘 최신 뉴스에서 가장 중요한 내용을 요약해서 100자 이내로 출력'
router_result = routerLLM(query)  # 사용자 질문을 라우터에 넣어 의도를 분리합니다.
tool_results = excute_tools(router_result)    # 각 도구 실행 결과를 모으는 리스트입니다.
final_result = final_llm_qween(query,tool_results)  # 최종 답변 생성 결과를 저장합니다.
print(final_result)

In [ ]:
# [코드 각주] 같은 최신 뉴스 요약 질문을 OpenAI 기반 final LLM 흐름으로 처리합니다.
query = '오늘 최신 뉴스에서 가장 중요한 내용을 요약해서 100자 이내로 출력'
router_result = routerLLM(query)  # 사용자 질문을 라우터에 넣어 의도를 분리합니다.
tool_results = excute_tools(router_result)    # 각 도구 실행 결과를 모으는 리스트입니다.
final_result = final_llm_openai(query,tool_results)  # 최종 답변 생성 결과를 저장합니다.
print(final_result)

# [8교시]

In [ ]:
# [코드 각주] xmltodict 패키지를 설치합니다.
# XML 응답을 딕셔너리로 바꾸는 패키지를 설치합니다.
!pip install xmltodict

In [ ]:
# [코드 각주] 공공데이터 날씨 API를 호출하고 XML 응답에서 기온과 강수 형태를 추출합니다.
# 날씨 조회 api
import requests  # HTTP API 요청을 보내기 위해 requests를 사용합니다.
from datetime import datetime
import xmltodict  # XML 데이터를 파이썬 딕셔너리로 변환합니다.
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
from dotenv import load_dotenv  # .env 파일의 API 키를 읽기 위해 dotenv를 사용합니다.
load_dotenv(override=True)  # .env 값을 현재 실행 환경에 반영합니다.

weather_api = os.getenv('WEATHER_API')  # 기상청 API 키를 환경 변수에서 가져옵니다.
print(f'weather_api : {weather_api[:10]}...')

def get_current_date():  # API 요청에 필요한 기준 날짜를 만듭니다.
    current_date = datetime.now().date()
    return current_date.strftime("%Y%m%d")

def get_current_hour():  # API 요청에 필요한 기준 시간을 만듭니다.
    now = datetime.now()
    return datetime.now().strftime("%H%M")

int_to_weather = {  # 강수 형태 코드를 사람이 읽는 날씨 표현으로 매핑합니다.
    "0": "맑음",
    "1": "비",
    "2": "비/눈",
    "3": "눈",
    "5": "빗방울",
    "6": "빗방울눈날림",
    "7": "눈날림"
}

def forecast(params):  # 기상청 API 응답에서 기온과 강수 형태를 추출합니다.
    url = 'http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst' # 초단기예보
    # 값 요청 (웹 브라우저 서버에서 요청 - url주소와 파라미터)
    res = requests.get(url, params)  # API URL에 파라미터를 붙여 GET 요청을 보냅니다.
    print(f'res : {res}')
    #XML -> 딕셔너리
    xml_data = res.text
    dict_data = xmltodict.parse(xml_data)  # XML 응답을 중첩 딕셔너리로 변환합니다.

    for item in dict_data['response']['body']['items']['item']:
        if item['category'] == 'T1H':
            temp = item['obsrValue']
        # 강수형태: 없음(0), 비(1), 비/눈(2), 눈(3), 빗방울(5), 빗방울눈날림(6), 눈날림(7)
        if item['category'] == 'PTY':
            sky = item['obsrValue']
            
    sky = int_to_weather[sky]
    
    return temp, sky

keys = weather_api

params ={'serviceKey' : keys,   # API 호출에 필요한 요청 파라미터를 정의합니다.
         'pageNo' : '1', 
         'numOfRows' : '10', 
         'dataType' : 'XML', 
         'base_date' : get_current_date(), 
         'base_time' : get_current_hour(), 
         'nx' : '55', 
         'ny' : '127' }

forecast(params)

In [ ]:
# [코드 각주] 서울시 실시간 도시데이터 API로 특정 장소의 혼잡도와 날씨를 조회합니다.
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
import requests  # HTTP API 요청을 보내기 위해 requests를 사용합니다.
import xmltodict  # XML 데이터를 파이썬 딕셔너리로 변환합니다.
from dotenv import load_dotenv  # .env 파일의 API 키를 읽기 위해 dotenv를 사용합니다.

# .env 파일 로드
load_dotenv(override=True)  # .env 값을 현재 실행 환경에 반영합니다.

# 1. 서울시 API 키 가져오기 (.env 파일에 SEOUL_API 변수 세팅 필요)
seoul_api_key = os.getenv('SEOUL_API')  # 서울시 API 키를 환경 변수에서 읽습니다.

def get_seoul_city_data(params):  # 서울시 도시데이터 API를 호출하는 함수입니다.
    # 2. URL 조립: 서울시 API는 params를 쿼리스트링이 아닌 URL 경로(Path)에 직접 넣습니다.
    url = f"http://openapi.seoul.go.kr:8088/{params['KEY']}/{params['TYPE']}/{params['SERVICE']}/{params['START_INDEX']}/{params['END_INDEX']}/{params['AREA_NM']}"  # API 요청을 보낼 최종 URL입니다.
    
    # URL 완성 후 GET 요청
    res = requests.get(url)  # API URL에 파라미터를 붙여 GET 요청을 보냅니다.
    print(f'응답 상태 코드 : {res.status_code}')
    
    if res.status_code != 200:
        print("API 요청에 실패했습니다. 키 값이나 네트워크를 확인하세요.")
        return None, None, None

    xml_data = res.text
    
    try:
        dict_data = xmltodict.parse(xml_data)  # XML 응답을 중첩 딕셔너리로 변환합니다.
        
        # 3. 정상 처리 확인 (결과 코드가 INFO-000 이면 정상)
        result_code = dict_data['SeoulRtd.citydata']['RESULT']['RESULT.CODE']
        if result_code != 'INFO-000':
            error_msg = dict_data['SeoulRtd.citydata']['RESULT']['RESULT.MESSAGE']
            print(f"[-] API 호출 오류: {error_msg}")
            return None, None, None

        # 4. 데이터 추출
        city_data = dict_data['SeoulRtd.citydata']['CITYDATA']  # 서울시 API 응답에서 실제 도시 데이터 영역을 꺼냅니다.
        
        # 인구 혼잡도 데이터
        congest_lvl = city_data['LIVE_PPLTN_STTS']['LIVE_PPLTN_STTS']['AREA_CONGEST_LVL']  # 실시간 인구 혼잡도 값을 추출합니다.
        
        # 날씨 데이터 (기온, 강수 형태)
        weather = city_data['WEATHER_STTS']['WEATHER_STTS']  # 서울시 API 응답의 날씨 데이터 영역입니다.
        temp = weather['TEMP']
        sky = weather['PRECPT_TYPE']
        
        return congest_lvl, temp, sky
        
    except Exception as e:
        print(f"데이터 파싱 중 에러 발생: {e}")
        return None, None, None


# 기존 코드처럼 파라미터 딕셔너리 셋업
params = {  # API 호출에 필요한 요청 파라미터를 정의합니다.
    'KEY': seoul_api_key, 
    'TYPE': 'xml', 
    'SERVICE': 'citydata', 
    'START_INDEX': '1', 
    'END_INDEX': '5', 
    'AREA_NM': '광화문·덕수궁'  # 다른 지역(예: 강남 MICE 관광특구 등)으로 변경 가능
}

print(f"요청 장소: {params['AREA_NM']} 데이터 조회 중...\n")

# 함수 호출
congest_lvl, temp, sky = get_seoul_city_data(params)

# 결과 출력
if congest_lvl and temp and sky:
    print("-" * 40)
    print(f"📍 장소: {params['AREA_NM']}")
    print(f"🚶‍♂️ 현재 혼잡도: {congest_lvl}")
    print(f"🌡️ 현재 기온: {temp}°C")
    print(f"☁️ 현재 날씨: {sky}")
    print("-" * 40)

In [ ]:
# [코드 각주] 서울시 도시데이터 조회 기능을 Agent가 호출할 수 있는 tool 함수로 감쌉니다.
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
import requests  # HTTP API 요청을 보내기 위해 requests를 사용합니다.
import xmltodict  # XML 데이터를 파이썬 딕셔너리로 변환합니다.

def seoul_city_data_tool(area_nm: str) -> dict:  # 서울시 데이터를 Agent 도구로 사용할 수 있게 감싼 함수입니다.
    """서울시 특정 장소의 혼잡도와 날씨를 조회하는 도구"""
    seoul_api_key = os.getenv('SEOUL_API')  # 서울시 API 키를 환경 변수에서 읽습니다.
    
    # LLM이 이상한 공백을 넣을 수도 있으니 정리
    area_nm = area_nm.strip() 
    
    url = f"http://openapi.seoul.go.kr:8088/{seoul_api_key}/xml/citydata/1/5/{area_nm}"  # API 요청을 보낼 최종 URL입니다.
    res = requests.get(url)  # API URL에 파라미터를 붙여 GET 요청을 보냅니다.
    
    if res.status_code != 200:
        return {"error": "API 요청에 실패했습니다."}

    try:
        dict_data = xmltodict.parse(res.text)  # XML 응답을 중첩 딕셔너리로 변환합니다.
        result_code = dict_data['SeoulRtd.citydata']['RESULT']['RESULT.CODE']
        
        if result_code != 'INFO-000':
            return {"error": dict_data['SeoulRtd.citydata']['RESULT']['RESULT.MESSAGE']}

        city_data = dict_data['SeoulRtd.citydata']['CITYDATA']  # 서울시 API 응답에서 실제 도시 데이터 영역을 꺼냅니다.
        congest_lvl = city_data['LIVE_PPLTN_STTS']['LIVE_PPLTN_STTS']['AREA_CONGEST_LVL']  # 실시간 인구 혼잡도 값을 추출합니다.
        weather = city_data['WEATHER_STTS']['WEATHER_STTS']  # 서울시 API 응답의 날씨 데이터 영역입니다.
        temp = weather['TEMP']
        sky = weather['PRECPT_TYPE']
        
        return {
            "장소": area_nm,
            "인구혼잡도": congest_lvl,
            "현재기온": f"{temp}도",
            "강수형태": sky
        }
    except Exception as e:
        return {"error": f"데이터를 찾을 수 없거나 파싱 오류 발생: {e}"}

In [ ]:
# [코드 각주] 서울데이터와 뉴스 의도를 함께 분리할 수 있는 routerLLM_v2를 작성합니다.
import json
from textwrap import dedent  # 멀티라인 프롬프트의 들여쓰기를 정리합니다.

def routerLLM_v2(query):  # 복합 질문을 여러 의도로 분리하는 LLM 라우터입니다.
    prompt = dedent(f"""
    사용자의 질문 의도를 분석하세요.
    질문에 포함된 의도가 여러 개이면 반드시 모든 의도를 각각 분리하여 JSON 배열로 출력하세요.

    question_type 별로 tool_query를 생성하세요.
    tool_query는 반드시 키워드 중심으로 생성하세요.

    지원 가능한 question_type 예시 및 규칙:
    - 뉴스 : api 검색할 수 있는 키워드 중심으로 추출
    - 주식 : 종목명만 추출
    - 서울데이터 : 사용자가 서울의 특정 장소(예: 광화문, 강남역 등)의 혼잡도, 사람 많은지, 날씨 등을 물어볼 때 사용. tool_query에는 '광화문·덕수궁', '강남역' 등 "정확한 장소명"만 추출하세요.

    [예시 입력]
    광화문·덕수궁 사람 많아? 그리고 관련 뉴스도 찾아줘

    [예시 출력]
    [
        {{
            "question_type": "서울데이터",
            "tool_query": "광화문·덕수궁",
            "reason": "특정 장소의 혼잡도 요청"
        }},
        {{
            "question_type": "뉴스",
            "tool_query": "광화문 덕수궁",
            "reason": "관련 뉴스 검색 요청"
        }}
    ]

    [질문]
    {query}

    반드시 유효한 JSON 배열만 출력하세요. (markdown 금지)
    """)

    result = llm.generate(prompt)
    return json.loads(result)

In [ ]:
# [코드 각주] routerLLM_v2 결과에 따라 뉴스 도구와 서울데이터 도구를 실행합니다.
def excute_tools_v2(router_result):  # 라우터 결과를 순회하며 필요한 도구를 실행합니다.
    tool_results = []  # 각 도구 실행 결과를 모으는 리스트입니다.
    for row in router_result:
        query_type = row.get('question_type')  # 현재 실행할 도구의 종류를 꺼냅니다.
        query = row.get('tool_query')
        
        print(f'🛠️ 도구 실행 중 : [{query_type}] -> 검색어: {query}')
        
        if query_type == '뉴스':        
            news_result = search_naver_news(query)  # 뉴스 도구 실행 결과를 저장합니다.
            tool_results.append({
                'question_type': '뉴스',
                'query': query,
                'result': news_result
            })
        elif query_type == '서울데이터':
            # 여기서 방금 만든 서울시 API 도구를 호출합니다!
            seoul_result = seoul_city_data_tool(query)
            tool_results.append({
                'question_type': '서울데이터',
                'query': query,
                'result': seoul_result
            })
            
    return tool_results

In [ ]:
# [코드 각주] OpenAI LLM 객체를 복구하고 연결 테스트를 수행합니다.
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
from dotenv import load_dotenv  # .env 파일의 API 키를 읽기 위해 dotenv를 사용합니다.
from openai import OpenAI  # OpenAI Chat Completions API를 사용하기 위한 클래스입니다.

# 1. 환경 변수 로드 (.env 파일에서 OPENAI_API_KEY 가져오기)
load_dotenv(override=True)  # .env 값을 현재 실행 환경에 반영합니다.
api_key = os.getenv('OPENAI_API_KEY')  # 환경 변수에서 OpenAI API 키를 가져옵니다.

if not api_key:  # API 키가 없을 때 잘못된 실행을 막는 방어 코드입니다.
    print("⚠️ OPENAI_API_KEY가 세팅되지 않았습니다. .env 파일을 다시 확인해 주세요.")
else:
    # 2. OpenAILLM 클래스 정의 (수업 시간에 사용하신 모델명 기준)
    class OpenAILLM:  # LLM 호출 코드를 재사용하기 쉽게 클래스로 묶습니다.
        def __init__(self, model: str = 'gpt-5.4-nano'): # 필요시 'gpt-4o-mini' 등으로 변경 가능
            self.client = OpenAI(api_key=api_key)  # API 요청을 보낼 클라이언트 객체를 저장합니다.
            self.model = model  # 사용할 모델명을 인스턴스 변수로 저장합니다.
            
        def generate(self, prompt: str) -> str:  # 프롬프트를 입력받아 LLM 응답 텍스트를 반환합니다.
            response = self.client.chat.completions.create(  # Chat Completions 방식으로 모델에 메시지를 전달합니다.
                model=self.model,
                messages=[  # system/user 역할 메시지를 모델에 전달하는 부분입니다.
                    {"role": "system", "content": "너는 사용자의 질문에 친절하고 정확하게 답변하는 시스템입니다."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,  # 생성 결과의 무작위성을 조절합니다. 0에 가까울수록 안정적입니다.
            )
            return response.choices[0].message.content  # 모델이 생성한 최종 텍스트만 꺼내 반환합니다.

    # 3. llm 객체 생성 (에러의 원인이었던 변수 부활!)
    llm = OpenAILLM()

    # 4. 정상 작동 확인용 테스트
    try:
        test_response = llm.generate("안녕? 연결 테스트 중이야. 짧게 인사해 줘.")
        print("✅ llm 객체 생성 및 연결 완료! API 테스트 응답:", test_response)
    except Exception as e:
        print("❌ API 호출 중 에러가 발생했습니다:", e)

In [ ]:
# [코드 각주] 도구 실행 결과를 최종 LLM에 전달해 종합 답변을 생성합니다.
import json
import os  # 환경 변수와 파일 경로 처리를 위해 os 모듈을 불러옵니다.
from openai import OpenAI  # OpenAI Chat Completions API를 사용하기 위한 클래스입니다.

# 1. 프롬프트 메시지 생성 함수 복구 (수업 코드와 동일)
def make_message(user_query:str, tool_results:list[dict]):  # 최종 LLM에 전달할 system/user 메시지를 구성합니다.
    prompt = f'''
    너는 다양한 도구의 결과를 종합해서 사용자 질문에 답변하는 ai assistant 입니다.

    [사용자질문]
    {user_query}

    [도구실행결과]
    {json.dumps(tool_results, ensure_ascii=False, indent=2)}

    [지침]
    - tool 결과를 기반으로 답변하세요
    - 필요한 경우 여러 tool 결과를 종합하세요
    - 지도 주식 날씨 검색 추천등 다양한 데이터가 포함될수 있습니다.
    - tool 결과내에 있는 내용에 우선순위를 높게해서 해당 데이터기반으로 답변하세요
    '''
    return [
        {"role":"system", "content":"너는 여러 tool결과를 조합해서 최종 답변을생성하는 ai agent 입니다."},
        {"role":"user", "content":prompt}
    ]
        
# 2. 최종 답변 생성 함수 복구
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))  # 최종 답변 생성에 사용할 OpenAI 클라이언트입니다.

def final_llm_openai(user_query:str, tool_results:list[dict]):      # 도구 결과를 OpenAI 모델로 종합해 최종 답변을 만듭니다.
    result = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages = make_message(user_query, tool_results),  # system/user 역할 메시지를 모델에 전달하는 부분입니다.
            temperature=0.3              # 생성 결과의 무작위성을 조절합니다. 0에 가까울수록 안정적입니다.
        )
    return result.choices[0].message.content

# 3. 모은 자료를 최종 출력
print("3. 최종 결과 집계 중...\n")
print("=" * 50)
final_result = final_llm_openai(user_query, tool_results)  # 최종 답변 생성 결과를 저장합니다.
print(final_result)
print("=" * 50)